In [6]:
%reload_ext autoreload
%autoreload 2

import io
import numpy as np
import matplotlib.pyplot as plt
import imageio.v3 as iio
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA

from vrAnalysis.files import literature_data_path
from vrAnalysis.external import pixease
from dimensionality_manuscript.comparison.claude_loop import gather_by_experiment_type, load

fpath = literature_data_path() / "CortexLab_ZebraNoise"
exp_paths = gather_by_experiment_type(fpath)

In [7]:
for key, val in exp_paths.items():
    print(f"{key}: ", "\n".join(f"  {p.name}" for p in val))

natural_images:    expcache_natural_images_BZ012_2024-11-19_11.npz
  expcache_natural_images_BZ012_2024-11-19_7.npz
  expcache_natural_images_BZ012_2024-11-20_6.npz
  expcache_natural_images_BZ012_2024-11-20_9.npz
  expcache_natural_images_BZ012_2024-11-21_6.npz
  expcache_natural_images_BZ012_2024-11-21_9.npz
  expcache_natural_images_BZ014_2025-04-02_12.npz
  expcache_natural_images_BZ014_2025-04-02_9.npz
  expcache_natural_images_BZ014_2025-04-16_2.npz
  expcache_natural_images_MS019_2025-02-11_10.npz
  expcache_natural_images_MS019_2025-02-11_13.npz
  expcache_natural_images_MS019_2025-02-12_10.npz
  expcache_natural_images_MS019_2025-02-12_7.npz
resting_state:    expcache_resting_state_BZ012_2024-11-19_10.npz
  expcache_resting_state_BZ012_2024-11-20_10.npz
  expcache_resting_state_BZ012_2024-11-20_5.npz
  expcache_resting_state_BZ012_2024-11-21_10.npz
  expcache_resting_state_BZ012_2024-11-21_5.npz
  expcache_resting_state_BZ014_2025-04-02_13.npz
  expcache_resting_state_BZ014_20

In [18]:
exp = load(exp_paths["natural_images"][3])

Skipping ball because it is not in the cache file


In [19]:
exp.summary()

=== Natural images ===
Type: natural_images (2p, pipeline "natural_images")
Mouse: BZ012
Session: 2024-11-20 exp 9 (BZ012_2024-11-20_9)
Duration: 25 repeats
Rig: b2 (1.3 zoom)
FOV: 7 x 512 x 512 (30 μm spacing)
Laser: 920nm (15% power)
Position: [1668.0, 63.5, 7356.0]
Notes:




In [ ]:
# Natural Images
exp.stimulus_timings()

,onset_time,offset_time,image
0,11.174,11.409,3
1,11.409,11.676,99
2,11.676,11.926,107
3,11.926,12.176,39
4,12.176,12.427,51
...,...,...,...
2963,758.124,758.375,67
2964,758.375,758.625,49
2965,758.625,758.891,78
2966,758.891,759.142,53


: 

In [58]:
exp.summary()

=== Full field drifting gratings location 2 ===
Type: full_field_drifting_grating (2p, pipeline "full_field_drifting_gratings")
Mouse: BZ014
Session: 2025-04-16 exp 14 (BZ014_2025-04-16_14)
Duration: 
Rig: b2 (1.3 zoom)
FOV: 7 x 512 x 512 (30 μm spacing)
Laser: 920nm (15% power)
Position: [2066.5, 881.5, -1397.0]
Notes:
    Gunk started todevelop in the mouse's eye during this experiment.  There was almost none in the previous experiment but in this one it was really bad.  Not sure if we can use these data.  The mouse doesn't look lik it can see at all.



In [11]:
exp.stimulus_timings()

IndexError: index -1 is out of bounds for axis 0 with size 0

In [64]:
# Drifting gratings
# Interval mean:
interval = list(zip(*(exp.stimulus_timings()["time_start"], exp.stimulus_timings()["time_stop"])))
exp.interval_mean(interval, "dspikes", cells=exp.cellinfo.iscell).shape

(336, 3336)

In [47]:
exp.timeseries((10, 30), "dspikes", cells=exp.cellinfo.iscell).shape

(5040, 200)

In [45]:
help(exp.timeseries)

Help on method timeseries in module vrAnalysis.external.pixease:

timeseries(intervals, measurement=None, dt=0.1, smooth=0.3, replacenan=False, prefilter=True, cells=None) method of vrAnalysis.external.pixease.EasyTimeseries instance
    Return a timeseries from this experiment.  Timeseries may be multidimensional.
    
    `intervals` can be either a tuple in the form of (start, end), or a list
    of tuples in the form (start, end).  If it is a list of tuples, this
    function returns a list of timeseries.
    
    `measurement` is the type of timeseries.  Valid values are:
    - "dspikes": Deconvolved fluorescence for all cells (2p only)
    - "f": Raw fluorescence for all cells (2p only)
    - "neuropil": Neuropil fluorescence for all cells (2p only)
    - "mean_pixels": Mean fluorescence of the frame (2p only)
    - "dff": Δf/f for all cells (2p only)
    - "pupil_size": Pupil size
    - "eye_x": Pupil x position
    - "eye_y": Pupil y position
    - "motion_energy": Motion energ